# Joint Burden + Cascade-Timing Calibration: Framework

Method-agnostic substrate for joint calibration of the India TB EMOD model
to (a) WHO/NTPS epidemic-burden targets AND (b) cascade-of-care timing
distributions. Used downstream by:

- `history_matching.ipynb` (primary method, GP emulator-based)
- `optuna_multiobjective.ipynb` (independent sanity check, NSGA-II)
- `pareto_methodological_figure.ipynb` (the methods-section centerpiece)

This notebook defines:
1. **Parameter space** with min/max bounds.
2. **Calibration targets** (point + tolerance + 'discrepancy' inflation).
3. **Latin-hypercube design generator** for batches of COMPS runs.
4. **Output parser** that walks a completed experiment folder and computes
   per-replicate burden + timing metrics.
5. **Implausibility / loss functions** (used by both methods).
6. **Calibration database** -- DataFrame of `{params, outputs, losses}` per point,
   saved to `calibration_db.parquet`. The DB is the shared state across waves
   and across methods.

## Compute budget

Each COMPS run with the current pop / replicate settings takes ~30-60 min.
Default plan: 4 waves x 15 design points x 3 replicates = ~180 sim-runs.
If that's too slow on CALCULON, drop replicates to 2 and reduce `x_Base_Population`
for waves 1-2 (rough scan), restoring full pop for wave 3-4 (refinement).


## 1. Imports and project setup


In [1]:
import os, sys, json
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Callable

import numpy as np
import pandas as pd
from scipy.stats import qmc
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from delay_decomposition_raw import (
    compute_delay_decomposition, apply_calibrated_loop_group, summarize_delays,
)
from recalibration_plan import (
    reclassify_loop_group_engagement_aware, EMPIRICAL_TARGETS,
)

CALIB_DB_PATH = PROJECT_ROOT / 'calibration_db.parquet'
print(f'Project root: {PROJECT_ROOT}')
print(f'Calibration DB: {CALIB_DB_PATH} (exists={CALIB_DB_PATH.exists()})')


Project root: C:\Users\young\OneDrive\Documents\GitHub\TB_India_DTK
Calibration DB: C:\Users\young\OneDrive\Documents\GitHub\TB_India_DTK\calibration_db.parquet (exists=False)


## 2. Parameter space

Five free parameters for the joint calibration. Each maps to a knob in the
simulation subsample dict (cell 24 of `01_India_TB_v5_simulation*.ipynb`).

| Param | Min | Max | Controls primarily |
|---|---|---|---|
| `Base_Infectivity_Constant` | 0.010 | 0.060 | burden (inc/prev/mort) |
| `TB Duration` (yr) | 1.5 | 3.0 | prevalence (duration tail) |
| `TB Presymptomatic Duration Years` | 0.15 | 0.50 | subclinical phase |
| `Care Seeking Fast Duration Days` | 30 | 180 | patient-delay (fast tail) |
| `Care Seeking Slow Duration Days` | 90 | 500 | patient-delay (slow tail) |

Other knobs (test sensitivities, treatment_fraction, drug cure) are held fixed
in this pass to keep the dimensionality tractable. If after history matching
the HS-delay axis remains stubbornly off, we add them as Wave 2 dimensions.


In [2]:
@dataclass
class Param:
    name: str         # human label
    subsample_key: str  # key in EMOD subsample dict
    lo: float
    hi: float
    transform: str = 'linear'  # 'linear' or 'log'

    def sample(self, u: float) -> float:
        """Map u in [0,1] to a parameter value."""
        if self.transform == 'log':
            lo, hi = np.log(self.lo), np.log(self.hi)
            return float(np.exp(lo + u * (hi - lo)))
        return float(self.lo + u * (self.hi - self.lo))

    def normalize(self, x: float) -> float:
        """Map a value back to [0,1] for emulator inputs."""
        if self.transform == 'log':
            return (np.log(x) - np.log(self.lo)) / (np.log(self.hi) - np.log(self.lo))
        return (x - self.lo) / (self.hi - self.lo)


PARAM_SPACE: list[Param] = [
    Param('beta',              'Base_Infectivity_Constant',          0.010, 0.060, 'log'),
    Param('tb_duration_yr',    'TB Duration',                        1.5,   3.0,   'linear'),
    Param('presymp_yr',        'TB Presymptomatic Duration Years',   0.15,  0.50,  'linear'),
    Param('cs_fast_d',         'Care Seeking Fast Duration Days',    30,    180,   'log'),
    Param('cs_slow_d',         'Care Seeking Slow Duration Days',    90,    500,   'log'),
]

PARAM_NAMES = [p.name for p in PARAM_SPACE]
print(f'Parameter space: {len(PARAM_SPACE)} dims')
for p in PARAM_SPACE:
    print(f'  {p.name:18s}  range=[{p.lo}, {p.hi}]  ({p.transform})')


Parameter space: 5 dims
  beta                range=[0.01, 0.06]  (log)
  tb_duration_yr      range=[1.5, 3.0]  (linear)
  presymp_yr          range=[0.15, 0.5]  (linear)
  cs_fast_d           range=[30, 180]  (log)
  cs_slow_d           range=[90, 500]  (log)


## 3. Calibration targets

Three target families:

1. **Burden** (3 targets): WHO 2020 incidence / NTPS 2019-21 prevalence / WHO 2020 mortality.
2. **Loop-group fractions** (3 targets): Subbaraman 2016 cascade distribution among engaged-cohort patients.
3. **Test counts** (4 targets): overall mean tests per active TB case (3-4 range)
   AND per-loop-group means so the cascade structure is pinned down, not just the marginal.

**Per-phase delays (subclinical, patient, HS) are NOT calibration targets.**
They're free model predictions that fall out of the loop structure + test
intervals. The parser still computes them and stores in the DB as
`DIAGNOSTIC_OUTPUTS`, but they don't enter the loss. This is methodologically
cleaner: empirical delay literature is biased low (recall, survivor, diagnosed-only),
while test counts are recorded directly in NTEP audits and meta-analyses.

Each target has:
- **`z`** = point estimate from empirical data
- **`sigma_obs`** = observation uncertainty (1-SD)
- **`sigma_disc`** = model-discrepancy term (Goldstein/Rougier idea -- additional
  variance acknowledging the model is a simplification of reality)
- **`group`** = 'burden' / 'fractions' / 'tests' (drives the loss decomposition)


In [3]:
@dataclass
class Target:
    name: str
    z: float
    sigma_obs: float
    sigma_disc: float
    group: str  # 'burden', 'fractions', or 'tests'

    @property
    def sigma_total(self) -> float:
        return np.sqrt(self.sigma_obs**2 + self.sigma_disc**2)


# === BURDEN TARGETS (WHO 2020 + NTPS 2019-21, per 100k) ====================
BURDEN_TARGETS = [
    Target('inc_per_100k_2020',  188.0, sigma_obs=25.0, sigma_disc=20.0, group='burden'),
    Target('prev_per_100k_2020', 304.0, sigma_obs=20.0, sigma_disc=30.0, group='burden'),
    Target('mort_per_100k_2020',  33.0, sigma_obs= 4.5, sigma_disc= 4.0, group='burden'),
]

# === LOOP-GROUP FRACTION TARGETS (Subbaraman 2016, engaged cohort) =========
FRACTION_TARGETS = [
    Target('frac_NoLooponly_engaged',   0.50, sigma_obs=0.08, sigma_disc=0.08, group='fractions'),
    Target('frac_TestLooponly_engaged', 0.35, sigma_obs=0.08, sigma_disc=0.08, group='fractions'),
    Target('frac_TestTxfail_engaged',   0.13, sigma_obs=0.05, sigma_disc=0.05, group='fractions'),
]

# === TEST-COUNT TARGETS (mean tests per active TB case, by loop group) =====
# Overall mean 3.5 (range 3-4 per Cazabon 2017 / NTEP audits).
# Per-group means reflect cascade structure: NoLoop ~1 single positive,
# TestLoop ~3 (1 pos + ~2 failures), TestTxfail ~4.5 (compound failures).
TEST_TARGETS = [
    Target('mean_tests_overall_engaged',        3.5, sigma_obs=0.6, sigma_disc=0.5, group='tests'),
    Target('mean_tests_NoLooponly_engaged',     1.0, sigma_obs=0.3, sigma_disc=0.3, group='tests'),
    Target('mean_tests_TestLooponly_engaged',   3.0, sigma_obs=0.7, sigma_disc=0.6, group='tests'),
    Target('mean_tests_TestTxfail_engaged',     4.5, sigma_obs=1.0, sigma_disc=0.8, group='tests'),
]

TARGETS: list[Target] = BURDEN_TARGETS + FRACTION_TARGETS + TEST_TARGETS
TARGET_NAMES = [t.name for t in TARGETS]

# Per-phase delay means are NOT calibration targets here -- they're free
# model predictions for separate validation. The parser still computes them
# and they're saved in the DB as diagnostic columns.
DIAGNOSTIC_OUTPUTS = [
    'subclinical_mean_d_engaged',
    'patient_delay_mean_d_engaged',
    'hs_delay_mean_d_engaged',
    'engagement_rate',
]

print(f'{len(TARGETS)} calibration targets:')
print(f'  burden   : {len(BURDEN_TARGETS)}')
print(f'  fractions: {len(FRACTION_TARGETS)}')
print(f'  tests    : {len(TEST_TARGETS)}')
print(f'\nDiagnostic outputs (saved but not calibrated): {len(DIAGNOSTIC_OUTPUTS)}')
df_targets = pd.DataFrame([asdict(t) for t in TARGETS])
df_targets['sigma_total'] = [t.sigma_total for t in TARGETS]
df_targets


10 calibration targets:
  burden   : 3
  fractions: 3
  tests    : 4

Diagnostic outputs (saved but not calibrated): 4


,name,z,sigma_obs,sigma_disc,group,sigma_total
0,inc_per_100k_2020,188.00,25.00,20.00,burden,32.015621
1,prev_per_100k_2020,304.00,20.00,30.00,burden,36.055513
2,mort_per_100k_2020,33.00,4.50,4.00,burden,6.020797
3,frac_NoLooponly_engaged,0.50,0.08,0.08,fractions,0.113137
4,frac_TestLooponly_engaged,0.35,0.08,0.08,fractions,0.113137
5,frac_TestTxfail_engaged,0.13,0.05,0.05,fractions,0.070711
6,mean_tests_overall_engaged,3.50,0.60,0.50,tests,0.781025
7,mean_tests_NoLooponly_engaged,1.00,0.30,0.30,tests,0.424264
8,mean_tests_TestLooponly_engaged,3.00,0.70,0.60,tests,0.921954
9,mean_tests_TestTxfail_engaged,4.50,1.00,0.80,tests,1.280625


## 4. Latin-hypercube design generator

Generates `n_points` parameter combinations using a scrambled LHS in [0,1]^d,
then maps each dimension to its parameter range. For wave > 1, accepts an
optional NROY mask (function from normalized space to {True, False}) and
rejection-samples inside it.


In [4]:
def generate_design(n_points: int,
                    nroy_mask: Optional[Callable[[np.ndarray], np.ndarray]] = None,
                    seed: int = 0,
                    max_oversample: int = 30) -> pd.DataFrame:
    """
    Latin-hypercube design over PARAM_SPACE. If `nroy_mask` is provided,
    samples are filtered to those passing the mask in normalized [0,1]^d.

    Returns a DataFrame with columns matching PARAM_NAMES.
    """
    rng = np.random.default_rng(seed)
    d = len(PARAM_SPACE)
    if nroy_mask is None:
        sampler = qmc.LatinHypercube(d=d, seed=rng)
        U = sampler.random(n=n_points)
    else:
        # Oversample LHS, then keep first n_points passing the NROY filter
        sampler = qmc.LatinHypercube(d=d, seed=rng)
        U_big = sampler.random(n=n_points * max_oversample)
        keep = nroy_mask(U_big)
        U = U_big[keep][:n_points]
        if len(U) < n_points:
            print(f'  WARNING: NROY too small -- only {len(U)} points found of {n_points} requested')
    rows = []
    for u in U:
        row = {p.name: p.sample(u[i]) for i, p in enumerate(PARAM_SPACE)}
        rows.append(row)
    return pd.DataFrame(rows)


# Smoke test
design_test = generate_design(n_points=5, seed=42)
print('Example design (no NROY filter):')
display(design_test.round(3))


Example design (no NROY filter):


,beta,tb_duration_yr,presymp_yr,cs_fast_d,cs_slow_d
0,0.010,2.727,0.439,38.425,181.504
1,0.039,1.501,0.167,119.190,311.120
2,0.017,2.347,0.343,129.081,108.667
3,0.058,1.821,0.289,47.036,420.029
4,0.028,2.572,0.373,64.792,174.497


## 5. Submission helpers (COMPS)

Two ways to use the framework:

**A. Recommended: batch sweep.** Take the design DataFrame, write a JSON file,
and have your modified version of `01_India_TB_v5_simulation*.ipynb` read it
as a sweep dimension. This way one COMPS experiment runs the whole wave.

**B. Manual: one experiment per design point.** Slower (more queue overhead)
but simpler. Use `print_subsample_dicts()` and paste each into the sim notebook.

Either way, after the runs complete, point `parse_experiment_dir()` at the
result folder(s) and the framework parses outputs into the DB.


In [5]:
def design_to_subsample_dicts(design: pd.DataFrame,
                              baseline_subsample: Optional[dict] = None) -> list[dict]:
    """
    Convert a design DataFrame into a list of EMOD `subsample` dicts that
    the simulation notebook can sweep over.
    """
    if baseline_subsample is None:
        # Default baseline copied from cell 24 of the v0 sim notebook (with
        # the recalibration knobs left as placeholders that the design fills in).
        baseline_subsample = {
            'ART Factor': 1.0,
            'CD4_aq_200_300_rel': 20.434,
            'CD4_aq_300_400_rel': 3.0,
            'CD4_aq_400_500_rel': 2.0,
            'CD4_aq_above_500': 1.0,
            'CD4_aq_below_200_rel': 54.55,
            'Death Fraction': 0.41,
            'Post_Infection_Acquisition_Multiplier': 0.40,
            'Primary HIV multiplier': 1.0,
            'Relative TB CD4 Infectiousness': 0.1,
            'TBHIV Duration': 1.5,
            'TB_Active_Presymptomatic_Infectivity_Multiplier': 0.1660,
            'TB_Fast_Progressor_Fraction_Adult': 0.072,
            'TB_Slow_Progressor_Rate': 1.01e-05,
        }
    out = []
    for _, row in design.iterrows():
        sample = dict(baseline_subsample)
        for p in PARAM_SPACE:
            sample[p.subsample_key] = float(row[p.name])
        out.append(sample)
    return out


def save_design_for_sweep(design: pd.DataFrame, wave: int) -> Path:
    """
    Save design as JSON; the simulation notebook reads this and adds it as a
    sweep dimension in `run_experiment()`. Output: design_wave{N}.json.
    """
    samples = design_to_subsample_dicts(design)
    out_path = PROJECT_ROOT / f'design_wave{wave}.json'
    with open(out_path, 'w') as f:
        json.dump({'wave': wave,
                   'n_points': len(samples),
                   'design_normalized': design.to_dict(orient='records'),
                   'subsamples': samples}, f, indent=2)
    print(f'Wrote {out_path}  ({len(samples)} design points)')
    return out_path


## 6. Output parser

Walks one experiment folder (or a list), loads `ReportEventRecorder.csv` and
`PropertyReport*.json`, computes the burden + timing metrics, and returns a
row of outputs aligned with `TARGET_NAMES`.

Assumes the experiment structure used by the existing v0/v1/v2 sim notebooks.


In [6]:
BURN_INITIAL  = 2 * 365         # match params.py
BURN_PREDOTS  = 145 * 365       # match params.py
SIM_DURATION  = BURN_INITIAL + BURN_PREDOTS + (38.0 + 15) * 365.0
ANALYSIS_WINDOW_DAYS = 10 * 365
YEAR_2020_DAY = BURN_INITIAL + BURN_PREDOTS + 23 * 365   # 2020 = DOTS_start + 23y
POP_SCALE = 1e5  # per 100k


def parse_experiment_dir(exp_dir: Path) -> dict:
    """
    Parse one COMPS experiment folder; return dict of {output_name: value}
    aligned with TARGET_NAMES + DIAGNOSTIC_OUTPUTS.
    """
    exp_dir = Path(exp_dir)
    sim_dirs = [s for s in exp_dir.iterdir()
                if s.is_dir() and (s / 'ReportEventRecorder.csv').exists()]
    if not sim_dirs:
        raise FileNotFoundError(f'No simulation outputs in {exp_dir}')

    events = []
    for i, sd in enumerate(sim_dirs):
        df = pd.read_csv(sd / 'ReportEventRecorder.csv')
        df['replicate'] = i
        events.append(df)
    event_df = pd.concat(events, ignore_index=True)

    # ── Burden metrics (year 2020 window) ──
    inc_2020, prev_2020, mort_2020 = _compute_burden(sim_dirs)

    # ── Cascade metrics from event log ──
    analysis_start = SIM_DURATION - ANALYSIS_WINDOW_DAYS
    event_recent = event_df[event_df['Time'] >= analysis_start].copy()
    delay_df = compute_delay_decomposition(event_recent)
    delay_df = reclassify_loop_group_engagement_aware(delay_df)

    # Estimated total tests per episode for engaged patients.
    # 1 confirmatory positive test + n_testneg + n_testdef for engaged;
    # 0 for NeverEngaged.
    delay_df['n_tests_episode'] = 1 + delay_df['n_testneg'].fillna(0) \
                                    + delay_df['n_testdef'].fillna(0)
    delay_df.loc[delay_df['loop_group_engaged'] == 'NeverEngaged',
                 'n_tests_episode'] = 0

    engaged = delay_df[delay_df['loop_group_engaged'] != 'NeverEngaged']
    n_total = max(len(delay_df), 1)
    n_eng = max(len(engaged), 1)

    def _grp_mean(col, grp):
        sub = engaged[engaged['loop_group_engaged'] == grp]
        return float(sub[col].mean()) if len(sub) else np.nan

    out = {
        # ── BURDEN ──
        'inc_per_100k_2020':            inc_2020,
        'prev_per_100k_2020':           prev_2020,
        'mort_per_100k_2020':           mort_2020,
        # ── LOOP-GROUP FRACTIONS (engaged-cohort base) ──
        'frac_NoLooponly_engaged':      float((engaged['loop_group_engaged'] == 'NoLooponly').sum() / n_eng),
        'frac_TestLooponly_engaged':    float((engaged['loop_group_engaged'] == 'TestLooponly').sum() / n_eng),
        'frac_TestTxfail_engaged':      float((engaged['loop_group_engaged'] == 'TestTxfail').sum() / n_eng),
        # ── TEST COUNTS (engaged-cohort base) ──
        'mean_tests_overall_engaged':       float(engaged['n_tests_episode'].mean()) if len(engaged) else np.nan,
        'mean_tests_NoLooponly_engaged':    _grp_mean('n_tests_episode', 'NoLooponly'),
        'mean_tests_TestLooponly_engaged':  _grp_mean('n_tests_episode', 'TestLooponly'),
        'mean_tests_TestTxfail_engaged':    _grp_mean('n_tests_episode', 'TestTxfail'),
        # ── DIAGNOSTIC OUTPUTS (not calibration targets) ──
        'subclinical_mean_d_engaged':   float(engaged['subclinical_days'].mean()) if len(engaged) else np.nan,
        'patient_delay_mean_d_engaged': float(engaged['patient_delay_days'].mean()) if len(engaged) else np.nan,
        'hs_delay_mean_d_engaged':      float(engaged['health_system_delay_days'].mean()) if len(engaged) else np.nan,
        'engagement_rate':              float(n_eng / n_total),
    }
    return out


def _compute_burden(sim_dirs: list[Path]) -> tuple[float, float, float]:
    """Year-2020 inc/prev/mort per 100k, averaged across replicates."""
    inc_vals, prev_vals, mort_vals = [], [], []
    for sd in sim_dirs:
        ic = sd / 'InsetChart.json'
        if not ic.exists():
            continue
        with open(ic) as f:
            data = json.load(f)
        channels = data['Channels']
        ts_step = data['Header']['Simulation_Timestep']
        step_2020 = int(YEAR_2020_DAY / ts_step)
        half_win = int(180 / ts_step)
        lo, hi = max(0, step_2020 - half_win), step_2020 + half_win
        try:
            pop = np.mean(channels['Statistical Population']['Data'][lo:hi])
        except KeyError:
            pop = 1.0
        for key, store in [('New_Active_TB_Infections', inc_vals),
                           ('Active_TB_Prevalence',     prev_vals),
                           ('Disease Deaths',           mort_vals)]:
            if key in channels:
                val_window = channels[key]['Data'][lo:hi]
                if key == 'Active_TB_Prevalence':
                    store.append(np.mean(val_window) * POP_SCALE)
                else:
                    per_year = np.sum(val_window) * (365.0 / (hi - lo) / ts_step)
                    store.append(per_year / pop * POP_SCALE if pop > 0 else np.nan)
    return (float(np.nanmean(inc_vals)) if inc_vals else np.nan,
            float(np.nanmean(prev_vals)) if prev_vals else np.nan,
            float(np.nanmean(mort_vals)) if mort_vals else np.nan)

print('Parser ready.')


Parser ready.


## 7. Implausibility + loss

**Standardized implausibility** for each target i:
$$ I_i(x) = \frac{|f_i(x) - z_i|}{\sigma_{\text{total},i}} $$

**Maximum implausibility** across targets is the history-matching criterion.
**Squared sum** is a smooth loss for Optuna / gradient-style methods.
**Group losses** (burden, timing) drive the Pareto-frontier figure.


In [7]:
def implausibility_vector(outputs: dict) -> dict:
    """Return dict of {target_name: I_i}."""
    out = {}
    for t in TARGETS:
        val = outputs.get(t.name, np.nan)
        if np.isnan(val):
            out[t.name] = np.inf
        else:
            out[t.name] = abs(val - t.z) / t.sigma_total
    return out


def implausibility_max(outputs: dict) -> float:
    return max(implausibility_vector(outputs).values())


def loss_squared(outputs: dict) -> float:
    """Sum of squared implausibilities -- smooth, used by Optuna."""
    iv = implausibility_vector(outputs)
    return float(sum(v**2 for v in iv.values()))


def group_losses(outputs: dict) -> dict:
    """Split squared-implausibility loss by target group."""
    iv = implausibility_vector(outputs)
    groups: dict[str, list[float]] = {'burden': [], 'fractions': [], 'tests': []}
    for t in TARGETS:
        groups[t.group].append(iv[t.name])
    out = {g: float(sum(v**2 for v in vs)) for g, vs in groups.items()}
    # Convenience: combined cascade loss (fractions + tests) for the Pareto y-axis
    out['cascade'] = out['fractions'] + out['tests']
    return out


## 8. Calibration database

Shared state across waves and across methods. Schema:
- One row per design point
- Columns: `wave`, `point_id`, `method`, `experiment_uid`, all PARAM_NAMES,
  all TARGET_NAMES, implausibility per target, `loss_burden`, `loss_timing`,
  `implausibility_max`.
- Persisted as `calibration_db.parquet`.


In [8]:
def make_row(wave: int, point_id: str, method: str, experiment_uid: str,
             params: dict, outputs: dict) -> dict:
    iv = implausibility_vector(outputs)
    gl = group_losses(outputs)
    row = {'wave': wave, 'point_id': point_id, 'method': method,
           'experiment_uid': experiment_uid}
    row.update({p.name: params.get(p.name, np.nan) for p in PARAM_SPACE})
    # Calibration targets
    row.update({t.name: outputs.get(t.name, np.nan) for t in TARGETS})
    # Diagnostic outputs (delays + engagement rate)
    row.update({name: outputs.get(name, np.nan) for name in DIAGNOSTIC_OUTPUTS})
    row.update({f'impl_{k}': v for k, v in iv.items()})
    row['loss_burden']    = gl['burden']
    row['loss_fractions'] = gl['fractions']
    row['loss_tests']     = gl['tests']
    row['loss_cascade']   = gl['cascade']    # combined cascade loss for Pareto y-axis
    row['implausibility_max'] = float(max(iv.values()))
    return row


def load_db() -> pd.DataFrame:
    if CALIB_DB_PATH.exists():
        return pd.read_parquet(CALIB_DB_PATH)
    cols = ['wave', 'point_id', 'method', 'experiment_uid'] \
        + PARAM_NAMES + TARGET_NAMES + DIAGNOSTIC_OUTPUTS \
        + [f'impl_{n}' for n in TARGET_NAMES] \
        + ['loss_burden', 'loss_fractions', 'loss_tests', 'loss_cascade',
           'implausibility_max']
    return pd.DataFrame(columns=cols)


def append_to_db(new_rows: list[dict]) -> pd.DataFrame:
    db = load_db()
    db = pd.concat([db, pd.DataFrame(new_rows)], ignore_index=True)
    db.to_parquet(CALIB_DB_PATH, index=False)
    print(f'  DB now has {len(db)} rows (wrote {CALIB_DB_PATH})')
    return db


## 9. Seed the DB with v0/v1/v2 manual runs

We already have three manual parameter points (the v0/v1/v2 sequence from the
recalibration discussion). Ingest them into the DB so history matching starts
with `wave=0` containing this prior knowledge.

Edit `MANUAL_RUNS` to point at the experiment folders on disk; the parser will
do the rest. If an experiment isn't downloaded yet, comment out that row.


In [9]:
# Edit these to match your local experiment folders.
MANUAL_RUNS = [
    {
        'point_id': 'v0_original',
        'experiment_uid': '68ce50cd-cc48-f111-92e4-000d3af5294c',
        'params': {'beta': 0.022, 'tb_duration_yr': 2.0, 'presymp_yr': 0.5,
                   'cs_fast_d': 160, 'cs_slow_d': 458},
    },
    # Uncomment once v1 / v2 pilot runs complete:
    # {'point_id': 'v1_delay_focused', 'experiment_uid': '<v1 UID>',
    #  'params': {'beta': 0.022, 'tb_duration_yr': 2.0, 'presymp_yr': 0.165,
    #             'cs_fast_d': 45, 'cs_slow_d': 120}},
    # {'point_id': 'v2_burden_compensated', 'experiment_uid': '<v2 UID>',
    #  'params': {'beta': 0.038, 'tb_duration_yr': 2.5, 'presymp_yr': 0.247,
    #             'cs_fast_d': 60, 'cs_slow_d': 180}},
]


def seed_db_from_manual_runs(runs: list[dict] = MANUAL_RUNS,
                              parse: bool = True) -> pd.DataFrame:
    rows = []
    for r in runs:
        exp_dir = PROJECT_ROOT / r['experiment_uid']
        if not exp_dir.exists():
            print(f'  SKIP {r["point_id"]}: {exp_dir} not found')
            continue
        try:
            outputs = parse_experiment_dir(exp_dir) if parse else {}
        except Exception as e:
            print(f'  PARSE FAIL {r["point_id"]}: {e}')
            continue
        row = make_row(wave=0, point_id=r['point_id'], method='manual',
                       experiment_uid=r['experiment_uid'],
                       params=r['params'], outputs=outputs)
        rows.append(row)
        print(f'  OK    {r["point_id"]:25s}  I_max={row["implausibility_max"]:.2f}  '
              f'L_burden={row["loss_burden"]:.2f}  L_cascade={row["loss_cascade"]:.2f}  '
              f'(fractions={row["loss_fractions"]:.2f}, tests={row["loss_tests"]:.2f})')
    if rows:
        return append_to_db(rows)
    return load_db()


# db = seed_db_from_manual_runs()
# display(db)


## 10. Wave driver (top-level workflow)

End-to-end pseudo-code for one wave. The two downstream notebooks each
implement step 4 differently (history matching uses GP implausibility;
Optuna uses NSGA-II ask-tell). All other steps are shared.

```
1. design = generate_design(n_points, nroy_mask=prev_wave_nroy)
2. save_design_for_sweep(design, wave=N)
3. -- COMPS sweep runs here, you launch it manually --
4. for each completed experiment:
       outputs = parse_experiment_dir(...)
       row     = make_row(wave=N, ..., outputs)
       db      = append_to_db([row])
5. db = load_db()
6. fit emulators on db (history matching) OR ask-tell trials (Optuna)
7. compute NROY mask for next wave
```

Done.
